# Ours (no Harmony) — screen-all duals + stack

Source: `attack.py`. Screen all 4 duals; stack encodings if 2+ confirm; farm best.

**How to submit:** Save & Run All, Internet OFF + GPU, then Submit.


In [ ]:
import sys
import os
import glob
from pathlib import Path

sys.argv = [sys.argv[0]]

dataset_root = None
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    break

print('dataset_root:', dataset_root)


In [ ]:
ATTACK_B64 = (
    "..."
    "Y2FuZGlkYXRlc1s6TUFYX0NBTkRJREFURVNdCg=="
)

from pathlib import Path
import base64

attack_bytes = base64.b64decode(ATTACK_B64)
out = Path('/kaggle/working/attack.py')
out.write_bytes(attack_bytes)
print(f'Wrote {out} ({len(attack_bytes)} bytes) [attack.py]')
print(out.read_text()[:500])


In [ ]:
import os
import pandas as pd

submission_path = '/kaggle/working/submission.csv'

submission = pd.DataFrame(
    {
        'Id': ['gpt_oss_public', 'gpt_oss_private', 'gemma_public', 'gemma_private'],
        'Score': [0.0, 0.0, 0.0, 0.0],
    }
)
submission.to_csv(submission_path, index=False)
print('Created', submission_path)
print(submission)

try:
    import kaggle_evaluation.jed_attack_134815.jed_attack_inference_server as server_module

    if hasattr(server_module, 'main'):
        server_module.main()
    elif hasattr(server_module, 'serve'):
        server_module.serve()
    else:
        server_cls = getattr(server_module, 'JEDAttackInferenceServer', None)
        if server_cls is not None:
            server_cls().serve()
    print('Inference server finished.')
except Exception as exc:
    print('Inference server not started in this context:', repr(exc))

if not os.path.exists(submission_path):
    submission.to_csv(submission_path, index=False)

final_submission = pd.read_csv(submission_path)
assert list(final_submission.columns) == ['Id', 'Score']
assert len(final_submission) == 4
print('submission.csv OK')
print(final_submission)
